# 霍夫变换直线与圆形检测实验：学生练习版

本练习版保留图像构造、边缘检测、结果显示和网络图像读取等辅助代码。
需要学生重点补全 **霍夫直线检测** 和 **霍夫圆形检测** 的核心算法。


## 1. 相关背景知识

### 1.1 霍夫变换的基本思想

霍夫变换是一种经典的形状检测方法。它的核心思想是：

1. 图像空间中的一个边缘点，可能属于许多条直线或许多圆。
2. 将每个边缘点映射到参数空间中进行投票。
3. 如果许多边缘点支持同一个形状参数，则该参数在累加器中会形成峰值。
4. 找到累加器中的峰值，即可得到图像中的直线或圆。

### 1.2 直线霍夫变换

直线可以用极坐标形式表示：

$$
\rho = x\cos\theta + y\sin\theta
$$

其中：

1. `rho` 表示原点到直线的垂直距离。
2. `theta` 表示直线法向量与 x 轴的夹角。
3. 每个边缘点 `(x, y)` 对所有候选 `theta` 计算 `rho`，并在累加器中投票。

当一组边缘点位于同一条直线上时，它们会在同一个 `(rho, theta)` 附近累积大量投票。

### 1.3 圆形霍夫变换

圆可以表示为：

$$
(x-a)^2+(y-b)^2=r^2
$$

其中：

1. `(a, b)` 是圆心。
2. `r` 是半径。

如果半径已知或半径范围有限，可以对每个边缘点 `(x, y)` 和候选半径 `r`，枚举圆周方向角 `alpha`，反推出可能的圆心：

$$
a = x - r\cos\alpha
$$

$$
b = y - r\sin\alpha
$$

若许多边缘点投票到同一个 `(a, b, r)`，说明图像中可能存在该圆。

## 2. 实验步骤

本实验按以下步骤完成：

1. 构造包含多条直线和多个圆的灰度测试图像。
2. 使用手写 Sobel 算子计算梯度幅值。
3. 根据梯度幅值阈值得到边缘点集合。
4. 对边缘点进行直线霍夫投票，得到 `(rho, theta)` 累加器。
5. 在直线累加器中寻找峰值，并将检测出的直线画回图像。
6. 对边缘点进行圆形霍夫投票，得到 `(r, y, x)` 累加器。
7. 在圆形累加器中寻找峰值，并将检测出的圆画回图像。

### 2.1 霍夫直线检测的完整流程

1. 读取或构造灰度图像，并将像素值归一化到 `[0, 1]` 范围。
2. 使用 Sobel 算子分别计算水平方向和垂直方向梯度。
3. 根据梯度幅值得到边缘图，边缘图中值为 1 的像素作为候选边缘点。
4. 设置直线参数范围：`theta` 通常取 `[-90°, 90°)`，`rho` 的范围由图像对角线长度决定。
5. 建立二维累加器 `accumulator[rho, theta]`，初始值全部为 0。
6. 遍历每一个边缘点 `(x, y)`，对每一个 `theta` 计算 `rho = x cos(theta) + y sin(theta)`。
7. 将计算得到的 `(rho, theta)` 对应累加器位置加 1，表示该边缘点为这条候选直线投了一票。
8. 在累加器中寻找局部峰值，票数越高说明越多边缘点支持该条直线。
9. 将峰值位置反变换为图像空间中的直线参数，并把检测到的直线画回原图。
10. 调整边缘阈值、`theta_step`、峰值阈值和非极大值抑制邻域，观察检测结果变化。

### 2.2 霍夫圆形检测的完整流程

1. 读取或构造灰度图像，完成平滑、梯度计算和边缘提取。
2. 根据任务需要设定候选半径集合 `radii`，例如 `18, 20, 22, ...`。
3. 建立三维累加器 `accumulator[r, y, x]`，其中 `(x, y)` 表示圆心位置，`r` 表示圆半径。
4. 遍历每一个边缘点 `(x, y)`。
5. 对每一个候选半径 `r`，再遍历一组角度 `angle`。
6. 根据圆方程反推可能圆心：`a = x - r cos(angle)`，`b = y - r sin(angle)`。
7. 如果圆心 `(a, b)` 位于图像范围内，就在三维累加器对应位置加 1。
8. 在三维累加器中寻找局部峰值，得到候选圆的圆心坐标和半径。
9. 使用空间邻域和半径邻域进行非极大值抑制，减少同一个圆附近出现多个重复检测结果。
10. 将检测出的圆画回图像，并比较真实图形与检测结果之间的偏差。
11. 调整半径范围、角度步长、峰值阈值和非极大值抑制邻域，分析圆形检测的准确率和计算量变化。

### 2.3 学生练习版补全顺序

建议按照下面的顺序补全代码：

1. 先补全 `hough_line_transform`：建立直线参数空间，遍历边缘点，计算 `rho`，完成二维累加器投票。
2. 再补全 `find_hough_peaks_2d`：根据 `threshold_ratio` 设置投票阈值，寻找最大峰值，并进行二维非极大值抑制。
3. 然后补全 `hough_circle_transform`：建立 `(radius, center_y, center_x)` 三维累加器，由边缘点和候选半径反推圆心并投票。
4. 最后补全 `find_hough_peaks_3d`：在三维累加器中寻找圆形峰值，并在半径和空间邻域内抑制重复检测。
5. 补全后先运行人工构造图像实验，再运行网络照片实验；如果结果过多或过少，再调整 `threshold_ratio`、`theta_step`、`angle_step`、候选半径范围和非极大值抑制邻域。


## 3. 导入基础库

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "Arial Unicode MS", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

## 4. 构造测试图像

In [ ]:
def draw_disk(image, y, x, radius, value):
    """
    在图像指定位置绘制小圆盘，用于加粗线条和圆周。
    """
    h, w = image.shape
    y0 = max(0, y - radius)
    y1 = min(h, y + radius + 1)
    x0 = max(0, x - radius)
    x1 = min(w, x + radius + 1)
    yy, xx = np.mgrid[y0:y1, x0:x1]
    mask = (yy - y) ** 2 + (xx - x) ** 2 <= radius ** 2
    image[y0:y1, x0:x1][mask] = value


def draw_line_segment(image, y0, x0, y1, x1, thickness=1, value=1.0):
    """
    在图像中绘制线段。
    """
    length = int(max(abs(y1 - y0), abs(x1 - x0))) + 1
    ys = np.linspace(y0, y1, length)
    xs = np.linspace(x0, x1, length)
    for y, x in zip(ys, xs):
        draw_disk(image, int(round(y)), int(round(x)), thickness, value)


def draw_circle_perimeter(image, cy, cx, radius, thickness=1, value=1.0):
    """
    在图像中绘制圆周。
    """
    angles = np.linspace(0, 2 * np.pi, 720, endpoint=False)
    for angle in angles:
        y = int(round(cy + radius * np.sin(angle)))
        x = int(round(cx + radius * np.cos(angle)))
        draw_disk(image, y, x, thickness, value)


def create_hough_test_image(size=160, random_state=4):
    """
    构造包含直线和圆形结构的灰度测试图像。
    """
    rng = np.random.default_rng(random_state)
    image = np.full((size, size), 0.08, dtype=np.float64)

    draw_line_segment(image, 18, 15, 130, 132, thickness=1, value=0.92)
    draw_line_segment(image, 32, 145, 142, 35, thickness=1, value=0.86)
    draw_line_segment(image, 118, 12, 118, 148, thickness=1, value=0.82)

    draw_circle_perimeter(image, 54, 54, 24, thickness=1, value=0.96)
    draw_circle_perimeter(image, 102, 108, 31, thickness=1, value=0.90)

    image += rng.normal(0, 0.035, size=image.shape)
    return np.clip(image, 0, 1)


image = create_hough_test_image()

plt.figure(figsize=(5, 5))
plt.imshow(image, cmap="gray", vmin=0, vmax=1)
plt.title("测试灰度图像")
plt.axis("off")
plt.show()

## 5. 手写 Sobel 边缘检测

In [ ]:
def sobel_gradient_magnitude(image):
    """
    使用 Sobel 算子计算梯度幅值。

    参数：
        image: 输入灰度图像

    返回：
        gradient: 梯度幅值图
    """
    image_float = image.astype(np.float64)
    padded = np.pad(image_float, pad_width=1, mode="edge")

    gx = (
        -padded[:-2, :-2] - 2 * padded[1:-1, :-2] - padded[2:, :-2]
        + padded[:-2, 2:] + 2 * padded[1:-1, 2:] + padded[2:, 2:]
    )
    gy = (
        -padded[:-2, :-2] - 2 * padded[:-2, 1:-1] - padded[:-2, 2:]
        + padded[2:, :-2] + 2 * padded[2:, 1:-1] + padded[2:, 2:]
    )

    gradient = np.sqrt(gx ** 2 + gy ** 2)
    return gradient


def threshold_edges(gradient, threshold_ratio=0.34):
    """
    根据梯度幅值阈值得到边缘图。
    """
    threshold = gradient.max() * threshold_ratio
    edges = gradient >= threshold
    return edges, threshold


gradient = sobel_gradient_magnitude(image)
edges, edge_threshold = threshold_edges(gradient, threshold_ratio=0.34)

print("边缘阈值：", edge_threshold)
print("边缘点数量：", int(edges.sum()))

plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.imshow(image, cmap="gray", vmin=0, vmax=1)
plt.title("原图")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(gradient, cmap="gray")
plt.title("Sobel 梯度幅值")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(edges, cmap="gray")
plt.title("边缘图")
plt.axis("off")

plt.tight_layout()
plt.show()

## 6. 手写霍夫直线检测（学生补全部分）

本节是关键考查内容。请学生补全直线参数空间投票和二维峰值检测。

需要重点理解：

1. 一个边缘点在参数空间中对应一条曲线。
2. 多个边缘点如果位于同一条图像直线上，它们会在相同或相近的 `(rho, theta)` 位置形成峰值。
3. `threshold_ratio` 是投票阈值比例，实际阈值为 `accumulator.max() * threshold_ratio`。


In [ ]:
def hough_line_transform(edges, theta_step=1):
    """
    手写霍夫直线变换。

    参数：
        edges: 二值边缘图，True 或 1 表示边缘点
        theta_step: theta 角度采样步长，单位为度；值越小，角度搜索越细，计算量越大

    返回：
        accumulator: 二维霍夫累加器，形状为 len(rhos) × len(thetas)
        rhos: rho 取值数组
        thetas: theta 取值数组，单位为弧度
    """
    h, w = edges.shape
    diag_len = int(np.ceil(np.sqrt(h ** 2 + w ** 2)))
    rhos = np.arange(-diag_len, diag_len + 1)
    theta_degrees = np.arange(-90, 90, theta_step)
    thetas = np.deg2rad(theta_degrees)

    accumulator = np.zeros((len(rhos), len(thetas)), dtype=np.int32)
    y_indices, x_indices = np.nonzero(edges)

    cos_values = np.cos(thetas)
    sin_values = np.sin(thetas)

    # TODO 1：遍历每一个边缘点 (y, x)。
    # TODO 2：对每一个 theta 计算 rho = x*cos(theta) + y*sin(theta)。
    # TODO 3：将 rho 转换为累加器中的 rho 下标。
    # TODO 4：将对应的 (rho_index, theta_index) 位置加 1，完成投票。
    # 提示：rho 的理论范围包含负数，因此转换为下标时需要加上 diag_len。
    raise NotImplementedError("请补全 hough_line_transform 中的直线霍夫投票过程")

    return accumulator, rhos, thetas


def find_hough_peaks_2d(accumulator, num_peaks=6, threshold_ratio=0.45, neighborhood_size=13):
    """
    在二维直线霍夫累加器中寻找局部峰值。

    参数：
        accumulator: 二维霍夫累加器
        num_peaks: 最多保留的直线数量
        threshold_ratio: 投票阈值比例，实际阈值为 accumulator.max() * threshold_ratio
        neighborhood_size: 非极大值抑制邻域大小；值越大，越容易合并相近直线

    返回：
        peaks: 峰值列表，每个元素为 (rho_index, theta_index, votes)
    """
    acc = accumulator.copy()
    peaks = []

    # TODO 5：计算最大票数 max_vote。如果 max_vote <= 0，直接返回空列表。
    # TODO 6：根据 threshold_ratio 计算投票阈值 threshold。
    # TODO 7：循环寻找累加器中的最大值位置。
    # TODO 8：如果最大值小于阈值，则停止搜索。
    # TODO 9：记录当前峰值 (rho_index, theta_index, votes)。
    # TODO 10：在 neighborhood_size 邻域内进行非极大值抑制，将附近位置清零。
    raise NotImplementedError("请补全 find_hough_peaks_2d 中的二维峰值检测过程")

    return peaks


line_accumulator, rhos, thetas = hough_line_transform(edges, theta_step=1)
line_peaks = find_hough_peaks_2d(line_accumulator, num_peaks=6, threshold_ratio=0.45, neighborhood_size=17)

print("检测到的直线峰值：")
for rho_idx, theta_idx, votes in line_peaks:
    print(f"rho={rhos[rho_idx]}, theta={np.rad2deg(thetas[theta_idx]):.1f}°, votes={votes}")

plt.figure(figsize=(7, 5))
plt.imshow(
    line_accumulator,
    cmap="hot",
    aspect="auto",
    extent=[np.rad2deg(thetas[0]), np.rad2deg(thetas[-1]), rhos[-1], rhos[0]],
)
plt.xlabel("theta / degree")
plt.ylabel("rho")
plt.title("直线霍夫累加器")
plt.colorbar(label="votes")
plt.show()


## 7. 显示直线检测结果

In [ ]:
def draw_detected_lines(ax, peaks, rhos, thetas, image_shape, color="lime"):
    """
    将检测到的霍夫直线画回图像。
    """
    h, w = image_shape
    xs = np.array([0, w - 1], dtype=np.float64)

    for rho_idx, theta_idx, votes in peaks:
        rho = rhos[rho_idx]
        theta = thetas[theta_idx]
        cos_t = np.cos(theta)
        sin_t = np.sin(theta)

        if abs(sin_t) > 1e-8:
            ys = (rho - xs * cos_t) / sin_t
            ax.plot(xs, ys, color=color, linewidth=2)
        else:
            x = rho / cos_t
            ax.plot([x, x], [0, h - 1], color=color, linewidth=2)


plt.figure(figsize=(6, 6))
plt.imshow(image, cmap="gray", vmin=0, vmax=1)
draw_detected_lines(plt.gca(), line_peaks, rhos, thetas, image.shape, color="lime")
plt.title("霍夫直线检测结果")
plt.xlim(0, image.shape[1] - 1)
plt.ylim(image.shape[0] - 1, 0)
plt.axis("off")
plt.show()

## 8. 手写霍夫圆形检测（学生补全部分）

本节是关键考查内容。请学生补全圆形参数空间投票和三维峰值检测。

需要重点理解：

1. 已知边缘点和候选半径时，可以根据不同角度反推出可能的圆心。
2. 圆形霍夫检测的参数空间是三维的：`(radius, center_y, center_x)`。
3. 圆形检测比直线检测计算量更大，因此半径范围和角度步长非常重要。


In [ ]:
def hough_circle_transform(edges, radii, angle_step=8):
    """
    手写霍夫圆形变换。

    参数：
        edges: 二值边缘图，True 或 1 表示边缘点
        radii: 候选半径数组；半径范围越大，计算量越大
        angle_step: 圆周角度采样步长，单位为度；值越小，圆心投票越密集，计算量越大

    返回：
        accumulator: 三维累加器，形状为 len(radii) × h × w
    """
    h, w = edges.shape
    accumulator = np.zeros((len(radii), h, w), dtype=np.int32)
    y_indices, x_indices = np.nonzero(edges)

    angles = np.deg2rad(np.arange(0, 360, angle_step))
    cos_values = np.cos(angles)
    sin_values = np.sin(angles)

    # TODO 11：遍历每一个候选半径 radius。
    # TODO 12：根据 radius 和 angles 计算圆心偏移量 dx、dy。
    # TODO 13：遍历每一个边缘点 (y, x)，反推可能圆心：
    #          center_x = x - dx
    #          center_y = y - dy
    # TODO 14：判断圆心是否在图像范围内。
    # TODO 15：对合法圆心位置在三维累加器 accumulator[radius_index, center_y, center_x] 中加 1。
    # 提示：如果使用数组下标批量投票，可以使用 np.add.at 处理重复圆心投票。
    raise NotImplementedError("请补全 hough_circle_transform 中的圆形霍夫投票过程")

    return accumulator


def find_hough_peaks_3d(accumulator, radii, num_peaks=4, threshold_ratio=0.42, spatial_neighborhood=14, radius_neighborhood=1):
    """
    在圆形霍夫三维累加器中寻找峰值。

    参数：
        accumulator: 三维圆形霍夫累加器
        radii: 候选半径数组
        num_peaks: 最多保留的圆数量
        threshold_ratio: 投票阈值比例，实际阈值为 accumulator.max() * threshold_ratio
        spatial_neighborhood: 圆心位置的非极大值抑制邻域大小
        radius_neighborhood: 半径方向的非极大值抑制邻域大小

    返回：
        peaks: 圆形峰值列表，每个元素为 (center_y, center_x, radius, votes)
    """
    acc = accumulator.copy()
    peaks = []

    # TODO 16：计算最大票数 max_vote。如果 max_vote <= 0，直接返回空列表。
    # TODO 17：根据 threshold_ratio 计算投票阈值 threshold。
    # TODO 18：循环寻找三维累加器中的最大值位置。
    # TODO 19：将最大值位置转换为 radius_index、center_y、center_x。
    # TODO 20：记录圆形检测结果 (center_y, center_x, radii[radius_index], votes)。
    # TODO 21：在半径方向和空间方向同时进行非极大值抑制，减少重复圆。
    raise NotImplementedError("请补全 find_hough_peaks_3d 中的三维峰值检测过程")

    return peaks


candidate_radii = np.arange(18, 38, 2)
circle_accumulator = hough_circle_transform(edges, candidate_radii, angle_step=8)
circle_peaks = find_hough_peaks_3d(
    circle_accumulator,
    candidate_radii,
    num_peaks=4,
    threshold_ratio=0.42,
    spatial_neighborhood=18,
    radius_neighborhood=1,
)

print("检测到的圆形峰值：")
for cy, cx, radius, votes in circle_peaks:
    print(f"center=({cy}, {cx}), radius={radius}, votes={votes}")


## 9. 显示圆形检测结果

In [ ]:
plt.figure(figsize=(6, 6))
ax = plt.gca()
ax.imshow(image, cmap="gray", vmin=0, vmax=1)

for cy, cx, radius, votes in circle_peaks:
    circle = plt.Circle((cx, cy), radius, color="cyan", fill=False, linewidth=2)
    ax.add_patch(circle)
    ax.scatter([cx], [cy], c="red", s=25)
    ax.text(cx + 3, cy + 3, f"r={radius}", color="yellow", fontsize=9)

plt.title("霍夫圆形检测结果")
plt.xlim(0, image.shape[1] - 1)
plt.ylim(image.shape[0] - 1, 0)
plt.axis("off")
plt.show()

## 10. 直线与圆形检测综合显示

In [ ]:
plt.figure(figsize=(7, 7))
ax = plt.gca()
ax.imshow(image, cmap="gray", vmin=0, vmax=1)

draw_detected_lines(ax, line_peaks, rhos, thetas, image.shape, color="lime")

for cy, cx, radius, votes in circle_peaks:
    circle = plt.Circle((cx, cy), radius, color="cyan", fill=False, linewidth=2)
    ax.add_patch(circle)
    ax.scatter([cx], [cy], c="red", s=25)

plt.title("霍夫直线与圆形检测综合结果")
plt.xlim(0, image.shape[1] - 1)
plt.ylim(image.shape[0] - 1, 0)
plt.axis("off")
plt.show()

## 11. 从网络地址读取照片并进行霍夫检测

前面的实验使用的是人工构造的简单图像，便于观察霍夫变换的投票过程。实际照片中会出现纹理、阴影、噪声和复杂背景，因此边缘点数量更多，检测结果也更容易受到参数影响。

本节从网络地址读取一张照片，然后复用前面已经手写实现的 Sobel 边缘检测、霍夫直线检测和霍夫圆形检测函数，对真实图像进行检测。

注意：`PHOTO_URL` 可以替换为其他图片地址。为了降低计算量，本节会将图像缩放到较小尺寸，并只选取梯度较强的一部分边缘点参与霍夫投票。


In [ ]:
from urllib.request import urlopen
from io import BytesIO


PHOTO_URL = "https://raw.githubusercontent.com/opencv/opencv/master/samples/data/smarties.png"


def read_network_image_as_gray(url):
    """
    从网络地址读取图像，并手动转换为灰度图像。

    参数：
        url: 网络图像地址

    返回：
        gray: 取值范围为 [0, 1] 的灰度图像
    """
    with urlopen(url, timeout=20) as response:
        image_bytes = response.read()

    raw_image = plt.imread(BytesIO(image_bytes))
    raw_image = np.asarray(raw_image)

    if raw_image.dtype == np.uint8:
        raw_image = raw_image.astype(np.float64) / 255.0
    else:
        raw_image = raw_image.astype(np.float64)

    if raw_image.ndim == 2:
        gray = raw_image
    else:
        rgb = raw_image[..., :3]
        gray = 0.299 * rgb[..., 0] + 0.587 * rgb[..., 1] + 0.114 * rgb[..., 2]

    gray_min = gray.min()
    gray_max = gray.max()
    gray = (gray - gray_min) / (gray_max - gray_min + 1e-12)
    return gray



def keep_strong_edge_points(gradient, base_edges, max_points=2200):
    """
    在边缘图中保留梯度较强的边缘点，减少真实照片上的霍夫投票计算量。
    """
    edge_positions = np.flatnonzero(base_edges.ravel())
    if len(edge_positions) <= max_points:
        return base_edges

    edge_values = gradient.ravel()[edge_positions]
    selected_order = np.argsort(edge_values)[-max_points:]
    selected_positions = edge_positions[selected_order]

    limited_edges = np.zeros_like(base_edges, dtype=bool)
    limited_edges.ravel()[selected_positions] = True
    return limited_edges


network_image = read_network_image_as_gray(PHOTO_URL)
network_gradient = sobel_gradient_magnitude(network_image)
network_edges_base, network_edge_threshold = threshold_edges(network_gradient, threshold_ratio=0.30)
network_edges = keep_strong_edge_points(network_gradient, network_edges_base, max_points=2200)

network_line_accumulator, network_rhos, network_thetas = hough_line_transform(network_edges, theta_step=2)
network_line_peaks = find_hough_peaks_2d(
    network_line_accumulator,
    num_peaks=8,
    threshold_ratio=0.38,
    neighborhood_size=15,
)

min_side = min(network_image.shape)
network_radii = np.arange(max(6, int(min_side * 0.04)), max(8, int(min_side * 0.18)), 3)
network_circle_accumulator = hough_circle_transform(network_edges, network_radii, angle_step=12)
network_circle_peaks = find_hough_peaks_3d(
    network_circle_accumulator,
    network_radii,
    num_peaks=8,
    threshold_ratio=0.45,
    spatial_neighborhood=16,
    radius_neighborhood=1,
)

print("网络图像地址：", PHOTO_URL)
print("缩放后图像尺寸：", network_image.shape)
print("边缘阈值：", network_edge_threshold)
print("参与投票的边缘点数量：", int(network_edges.sum()))

print("\n网络图像中的直线检测结果：")
for rho_idx, theta_idx, votes in network_line_peaks:
    print(f"rho={network_rhos[rho_idx]}, theta={np.rad2deg(network_thetas[theta_idx]):.1f}°, votes={votes}")

print("\n网络图像中的圆形检测结果：")
for cy, cx, radius, votes in network_circle_peaks:
    print(f"center=({cy}, {cx}), radius={radius}, votes={votes}")

plt.figure(figsize=(15, 10))

plt.subplot(2, 3, 1)
plt.imshow(network_image, cmap="gray", vmin=0, vmax=1)
plt.title("网络照片灰度图")
plt.axis("off")

plt.subplot(2, 3, 2)
plt.imshow(network_gradient, cmap="gray")
plt.title("Sobel 梯度幅值")
plt.axis("off")

plt.subplot(2, 3, 3)
plt.imshow(network_edges, cmap="gray")
plt.title("强边缘点")
plt.axis("off")

ax = plt.subplot(2, 3, 4)
ax.imshow(network_image, cmap="gray", vmin=0, vmax=1)
draw_detected_lines(ax, network_line_peaks, network_rhos, network_thetas, network_image.shape, color="lime")
ax.set_title("网络照片直线检测")
ax.set_xlim(0, network_image.shape[1] - 1)
ax.set_ylim(network_image.shape[0] - 1, 0)
ax.axis("off")

ax = plt.subplot(2, 3, 5)
ax.imshow(network_image, cmap="gray", vmin=0, vmax=1)
for cy, cx, radius, votes in network_circle_peaks:
    circle = plt.Circle((cx, cy), radius, color="cyan", fill=False, linewidth=2)
    ax.add_patch(circle)
    ax.scatter([cx], [cy], c="red", s=18)
ax.set_title("网络照片圆形检测")
ax.set_xlim(0, network_image.shape[1] - 1)
ax.set_ylim(network_image.shape[0] - 1, 0)
ax.axis("off")

ax = plt.subplot(2, 3, 6)
ax.imshow(network_image, cmap="gray", vmin=0, vmax=1)
draw_detected_lines(ax, network_line_peaks, network_rhos, network_thetas, network_image.shape, color="lime")
for cy, cx, radius, votes in network_circle_peaks:
    circle = plt.Circle((cx, cy), radius, color="cyan", fill=False, linewidth=2)
    ax.add_patch(circle)
    ax.scatter([cx], [cy], c="red", s=18)
ax.set_title("网络照片综合检测")
ax.set_xlim(0, network_image.shape[1] - 1)
ax.set_ylim(network_image.shape[0] - 1, 0)
ax.axis("off")

plt.tight_layout()
plt.show()


## 12. 实验小结

本练习版要求学生重点补全霍夫直线检测和霍夫圆形检测的核心算法。

需要掌握的重点：

1. 霍夫变换将图像空间中的边缘点映射到参数空间中投票。
2. 直线检测使用 `(rho, theta)` 参数空间，核心是由边缘点计算不同角度下的 `rho` 并投票。
3. 圆形检测使用 `(center_y, center_x, radius)` 参数空间，核心是由边缘点和候选半径反推圆心并投票。
4. 累加器中的峰值对应图像中可能存在的形状。
5. `threshold_ratio` 是峰值检测中的投票阈值比例，会直接影响检测结果数量。
6. 非极大值抑制可以减少同一条直线或同一个圆附近的重复检测结果。
7. 本实验没有使用 OpenCV 或 scikit-image 的现成检测函数，投票和峰值检测需要学生自己实现。

思考题：

1. 为什么边缘点越多，霍夫变换计算量越大？
2. 直线霍夫变换为什么使用 `(rho, theta)` 而不是斜率截距形式？
3. 圆形检测为什么比直线检测计算量更大？
4. 如果圆半径未知范围很大，如何降低圆形霍夫变换的计算量？
5. 投票阈值 `threshold_ratio` 太大或太小时，检测结果会出现什么变化？
